# DoRA A/B Comparison — Sylhet Dialect → English

Tests whether DoRA improves translation quality for Sylhet.
Embeddings are frozen in both trials so **DoRA is the only variable**.

| Trial | `use_dora` | Description |
|-------|-----------|-------------|
| A | `False` | Baseline — plain LoRA, frozen embeddings |
| B | `True`  | DoRA only, frozen embeddings |

Best config from Sylhet hp-tuning is pre-loaded. Run A then B, compare BLEU/chrF, decide.


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║              ⚠️  AUTO-RUN GUARD  ⚠️                          ║
# ║  Delete or skip this cell, then run cells one at a time.    ║
# ╚══════════════════════════════════════════════════════════════╝
import sys
print("⛔ Auto-run blocked. Delete this cell and run manually with Shift+Enter.")
sys.exit("Auto-run guard triggered.")

In [15]:
!pip install transformers==5.2.0 peft==0.18.1 datasets sacrebleu \
             rouge-score nltk jiwer sentencepiece accelerate \
             evaluate --quiet

In [16]:
# ── Configuration ────────────────────────────────────────────────────────────

DIALECT_NAME    = "Noakhali"
SRC_LANG        = "bng_noakhali"
TGT_LANG        = "eng_Latn"

# ── File paths ────────────────────────────────────────────────────────────────
BASE_MODEL_PATH = "/kaggle/input/datasets/farhanaadri/nllb-dialect-model"

# Training CSV
DIALECT_CSV     = "/kaggle/input/datasets/abonykamal/dataset-translated-train/noakhali_dialect_final.csv"
DIALECT_COL     = "dialect_speech"
ENGLISH_COL     = "English_translation"

# Validation CSV (Vashantor format)
VAL_CSV         = "/kaggle/input/datasets/abonykamal/final-validation/noakhali_validation.csv"
DIALECT_COL_VAL = "noakhali_bangla_speech "   # note: trailing space — check your CSV header
ENGLISH_COL_VAL = "english_speech"

# Standard Bangla anchor (prevents catastrophic forgetting; set BANGLA_CSV=None to skip)
BANGLA_CSV          = DIALECT_CSV
BANGLA_DIALECT_COL  = "Standard_Bangla"
BANGLA_ENGLISH_COL  = "English_translation"
BANGLA_ANCHOR_FRAC  = 0.10

# ── Best config from Sylhet hp-tuning ────────────────────────────────────────
LEARNING_RATE        = 0.0005          # 5e-4
LR_SCHEDULER         = "linear"
LORA_TARGET_MODULES  = ["q_proj", "v_proj", "k_proj", "out_proj"]
LORA_R               = 16
LORA_ALPHA           = 32
LORA_DROPOUT         = 0.05

NUM_BEAMS            = 5
LENGTH_PENALTY       = .8

# ── Training budget (5 epochs is enough to compare A vs B) ───────────────────
NUM_EPOCHS      = 5
BATCH_SIZE      = 16
GRAD_ACCUM      = 2
WARMUP_STEPS    = 100
MAX_SOURCE_LEN  = 128
MAX_TARGET_LEN  = 128
WEIGHT_DECAY    = 0.01
FP16            = True
RANDOM_SEED     = 42

OUTPUT_DIR = f"/kaggle/working/{DIALECT_NAME.lower()}_dora_ab"

print(f"Dialect       : {DIALECT_NAME}")
print(f"SRC tag       : {SRC_LANG}")
print(f"Base model    : {BASE_MODEL_PATH}")
print(f"Train CSV     : {DIALECT_CSV}")
print(f"Val CSV       : {VAL_CSV}")
print(f"LR            : {LEARNING_RATE}  |  scheduler: {LR_SCHEDULER}")
print(f"Target modules: {LORA_TARGET_MODULES}")
print(f"Beams         : {NUM_BEAMS}  |  length_penalty: {LENGTH_PENALTY}")
print(f"Epochs/trial  : {NUM_EPOCHS}")


Dialect       : Noakhali
SRC tag       : bng_noakhali
Base model    : /kaggle/input/datasets/farhanaadri/nllb-dialect-model
Train CSV     : /kaggle/input/datasets/abonykamal/dataset-translated-train/noakhali_dialect_final.csv
Val CSV       : /kaggle/input/datasets/abonykamal/final-validation/noakhali_validation.csv
LR            : 0.0005  |  scheduler: linear
Target modules: ['q_proj', 'v_proj', 'k_proj', 'out_proj']
Beams         : 5  |  length_penalty: 0.8
Epochs/trial  : 5


In [17]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import re
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional

import torch
from torch.utils.data import Dataset

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from transformers import (
    NllbTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    PeftModel,
)
import sacrebleu
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from jiwer import wer, cer

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
n_gpus = torch.cuda.device_count()
print(f"Device  : {device}")
print(f"Num GPUs: {n_gpus}")
if device == "cuda":
    for i in range(n_gpus):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

Device  : cuda
Num GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [18]:
# ── Dialect-safe normalization ────────────────────────────────────────────────
#
# Applies ONLY whitespace / punctuation / zero-width-char normalization.
# Does NOT apply csebuet phoneme normalization which would damage dialect features.

_PUNCT_MAP = str.maketrans({
    "\u201c": '"', "\u201d": '"',   # curly double quotes
    "\u2018": "'", "\u2019": "'",   # curly single quotes
    "\u2013": "-", "\u2014": "-",   # en-dash, em-dash
    "\u2026": "...",                 # ellipsis
    "\u00a0": " ",                   # non-breaking space
    "\u200b": "",                    # zero-width space
    "\u200c": "",                    # zero-width non-joiner
    "\u200d": "",                    # zero-width joiner
    "\ufeff": "",                    # BOM
})

def normalize_dialect(text: str) -> str:
    """Safe normalization that preserves dialectal features."""
    if not isinstance(text, str):
        return ""
    text = text.translate(_PUNCT_MAP)
    text = re.sub(r"[\u200b-\u200f\ufeff]", "", text)
    text = re.sub(r" +", " ", text)
    text = text.strip()
    return text

# Quick smoke test
test = "\u200b\u201cSylhet test\u201d  "
print(f"Before: {repr(test)}")
print(f"After : {repr(normalize_dialect(test))}")
print("Normalization ready.")


Before: '\u200b“Sylhet test”  '
After : '"Sylhet test"'
Normalization ready.


In [19]:
# ── Load training and validation data ────────────────────────────────────────

import pandas as pd

df = pd.read_csv(DIALECT_CSV)
print(f"Loaded {len(df)} rows from {DIALECT_CSV}")
print(f"Columns: {list(df.columns)}")

df = df[[DIALECT_COL, ENGLISH_COL]].dropna().reset_index(drop=True)
df = df[df[DIALECT_COL].str.strip().ne("") & df[ENGLISH_COL].str.strip().ne("")]
print(f"After dropping nulls/empty: {len(df)} rows")

df[DIALECT_COL] = df[DIALECT_COL].apply(normalize_dialect)
df[ENGLISH_COL] = df[ENGLISH_COL].apply(lambda x: x.strip() if isinstance(x, str) else x)

df_train = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# Load validation CSV
df_val_raw = pd.read_csv(VAL_CSV)
print(f"\nVal CSV columns: {list(df_val_raw.columns)}")
df_val = df_val_raw[[DIALECT_COL_VAL, ENGLISH_COL_VAL]].dropna().reset_index(drop=True)
df_val[DIALECT_COL_VAL] = df_val[DIALECT_COL_VAL].apply(normalize_dialect)
df_val[ENGLISH_COL_VAL] = df_val[ENGLISH_COL_VAL].apply(lambda x: x.strip() if isinstance(x, str) else x)

print(f"\nTrain samples : {len(df_train)}")
print(f"Val samples   : {len(df_val)}")
print(f"\nSample row:")
print(f"  Source : {df_train[DIALECT_COL].iloc[0]}")
print(f"  Target : {df_train[ENGLISH_COL].iloc[0]}")


Loaded 8235 rows from /kaggle/input/datasets/abonykamal/dataset-translated-train/noakhali_dialect_final.csv
Columns: ['Standard_Bangla', 'dialect_speech', 'English_translation']
After dropping nulls/empty: 8235 rows

Val CSV columns: ['bangla_speech ', 'banglish_speech ', 'noakhali_bangla_speech ', 'noakhali_banglish_speech ', 'region_name ', 'english_speech']

Train samples : 8235
Val samples   : 75

Sample row:
  Source : অবিলম্বে বাতিল করা হোক এই অভিশাপ।
  Target : This curse must be revoked immediately.


In [20]:
# ── Standard Bangla anchor data (prevents catastrophic forgetting) ────────────

df_bangla_train = None

if BANGLA_CSV is not None:
    df_bn = pd.read_csv(BANGLA_CSV)
    df_bn = df_bn[[BANGLA_DIALECT_COL, BANGLA_ENGLISH_COL]].dropna().reset_index(drop=True)
    df_bn.columns = [DIALECT_COL, ENGLISH_COL]

    n_anchor = int(len(df_train) * BANGLA_ANCHOR_FRAC)
    df_bangla_train = df_bn.sample(
        n=min(n_anchor, len(df_bn)), random_state=RANDOM_SEED
    ).reset_index(drop=True)

    print(f"Standard Bangla anchor samples: {len(df_bangla_train)}")
else:
    print("No Bangla anchor data configured (BANGLA_CSV is None). Skipping.")


Standard Bangla anchor samples: 823


In [21]:
# ── Load tokenizer and model ──────────────────────────────────────────────────

print(f"Loading tokenizer from: {BASE_MODEL_PATH}")
tokenizer = NllbTokenizer.from_pretrained(BASE_MODEL_PATH, use_fast=False)
print(f"  Vocab size: {len(tokenizer)}")

# Verify our dialect tag is present
src_tag_id = tokenizer.convert_tokens_to_ids(SRC_LANG)
assert src_tag_id != tokenizer.unk_token_id, (
    f"Source lang tag '{SRC_LANG}' not found in tokenizer! "
    f"Make sure you're loading from the dialect-tagged model, not the original NLLB."
)
tgt_tag_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
print(f"  {SRC_LANG} → token_id {src_tag_id} ✅")
print(f"  {TGT_LANG} → token_id {tgt_tag_id} ✅")

print(f"\nLoading model from: {BASE_MODEL_PATH}")
from transformers import AutoConfig

# Load and fix config before loading model
config = AutoConfig.from_pretrained(BASE_MODEL_PATH)

# Fix: the tied_weights_keys field may be a list instead of dict in saved config
if hasattr(config, 'tied_weights_keys') and isinstance(config.tied_weights_keys, list):
    pass  # this is fine, just suppress

model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_PATH,
    config=config,
    dtype=torch.float16 if FP16 else torch.float32,
    ignore_mismatched_sizes=True,
)
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading tokenizer from: /kaggle/input/datasets/farhanaadri/nllb-dialect-model
  Vocab size: 256209
  bng_noakhali → token_id 256206 ✅
  eng_Latn → token_id 256047 ✅

Loading model from: /kaggle/input/datasets/farhanaadri/nllb-dialect-model


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

  Parameters: 615,076,864


In [22]:
# ── Dataset class ────────────────────────────────────────────────────────────

import random
from torch.utils.data import Dataset

class DialectDataset(Dataset):
    def __init__(self, sources, targets, tokenizer, src_lang, tgt_lang,
                 max_source_len=128, max_target_len=128):
        self.sources        = sources
        self.targets        = targets
        self.tokenizer      = tokenizer
        self.src_lang       = src_lang
        self.tgt_lang       = tgt_lang
        self.max_source_len = max_source_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.sources)

    def __getitem__(self, idx):
        source = str(self.sources[idx])
        target = str(self.targets[idx])

        self.tokenizer.src_lang = self.src_lang
        model_inputs = self.tokenizer(
            source, max_length=self.max_source_len, truncation=True, padding=False,
        )

        self.tokenizer.src_lang = self.tgt_lang
        labels = self.tokenizer(
            target, max_length=self.max_target_len, truncation=True, padding=False,
        )
        self.tokenizer.src_lang = self.src_lang

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs


# ── Build train/val datasets ──────────────────────────────────────────────────

train_sources = df_train[DIALECT_COL].tolist()
train_targets = df_train[ENGLISH_COL].tolist()

if df_bangla_train is not None:
    train_sources += df_bangla_train[DIALECT_COL].tolist()
    train_targets += df_bangla_train[ENGLISH_COL].tolist()
    print(f"Combined: {len(df_train)} Sylhet + {len(df_bangla_train)} Bangla anchor")

combined = list(zip(train_sources, train_targets))
random.seed(RANDOM_SEED)
random.shuffle(combined)
train_sources, train_targets = zip(*combined)

train_dataset = DialectDataset(
    list(train_sources), list(train_targets),
    tokenizer, SRC_LANG, TGT_LANG, MAX_SOURCE_LEN, MAX_TARGET_LEN
)

val_dataset = DialectDataset(
    df_val[DIALECT_COL_VAL].tolist(), df_val[ENGLISH_COL_VAL].tolist(),
    tokenizer, SRC_LANG, TGT_LANG, MAX_SOURCE_LEN, MAX_TARGET_LEN
)

print(f"\nTrain dataset : {len(train_dataset)} samples")
print(f"Val dataset   : {len(val_dataset)} samples")


Combined: 8235 Sylhet + 823 Bangla anchor

Train dataset : 9058 samples
Val dataset   : 75 samples


In [23]:
# ── A/B Grid Definition ──────────────────────────────────────────────────────
# Only testing DoRA on/off. Embeddings stay frozen in both trials.

GRID = [
    {
        "label"               : "A_baseline",
        "use_dora"            : False,
        "unfreeze_embeddings" : False,
        "description"         : "LoRA only — no DoRA, frozen embeddings",
    },
    {
        "label"               : "B_dora",
        "use_dora"            : True,
        "unfreeze_embeddings" : False,
        "description"         : "DoRA — frozen embeddings",
    },
]

DORA_RESULTS = []

print(f"Grid: {len(GRID)} trials x {NUM_EPOCHS} epochs each")
for g in GRID:
    print(f"  [{g['label']}]  use_dora={g['use_dora']}  — {g['description']}")


Grid: 2 trials x 5 epochs each
  [A_baseline]  use_dora=False  — LoRA only — no DoRA, frozen embeddings
  [B_dora]  use_dora=True  — DoRA — frozen embeddings


In [24]:
# ── Helper: run one trial ─────────────────────────────────────────────────────

import time
import numpy as np

def run_trial(cfg: dict) -> dict:
    """
    Train one adapter variant (A or B) and evaluate it.
    cfg keys: label, use_dora, unfreeze_embeddings, description
    """
    label               = cfg["label"]
    use_dora            = cfg["use_dora"]
    unfreeze_embeddings = cfg["unfreeze_embeddings"]

    print()
    print("=" * 65)
    print(f"  TRIAL [{label}]  —  {cfg['description']}")
    print(f"  use_dora={use_dora}   unfreeze_embeddings={unfreeze_embeddings}")
    print("=" * 65)

    from transformers import AutoConfig, AutoModelForSeq2SeqLM
    from peft import get_peft_model, LoraConfig, TaskType

    trial_config = AutoConfig.from_pretrained(BASE_MODEL_PATH)
    trial_model  = AutoModelForSeq2SeqLM.from_pretrained(
        BASE_MODEL_PATH,
        config=trial_config,
        dtype=torch.float16 if FP16 else torch.float32,
        ignore_mismatched_sizes=True,
    )

    lora_cfg = LoraConfig(
        task_type      = TaskType.SEQ_2_SEQ_LM,
        r              = LORA_R,
        lora_alpha     = LORA_ALPHA,
        lora_dropout   = LORA_DROPOUT,
        target_modules = LORA_TARGET_MODULES,
        bias           = "none",
        use_dora       = use_dora,
    )
    trial_model = get_peft_model(trial_model, lora_cfg)

    emb_params_unfrozen = 0
    if unfreeze_embeddings:
        for name, param in trial_model.named_parameters():
            if "embed_tokens" in name or "shared" in name:
                param.requires_grad = True
                emb_params_unfrozen += param.numel()
        print(f"  Unfroze embedding params: {emb_params_unfrozen:,}")

    trainable = sum(p.numel() for p in trial_model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in trial_model.parameters())
    print(f"  Trainable: {trainable:,}  ({100*trainable/total:.3f}%)")

    trial_out  = f"{OUTPUT_DIR}/trial_{label}"
    trial_args = Seq2SeqTrainingArguments(
        output_dir                  = trial_out,
        num_train_epochs            = NUM_EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        learning_rate               = LEARNING_RATE,
        warmup_steps                = WARMUP_STEPS,
        weight_decay                = WEIGHT_DECAY,
        lr_scheduler_type           = LR_SCHEDULER,
        fp16                        = FP16,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        predict_with_generate       = False,
        logging_steps               = 50,
        save_total_limit            = 1,
        report_to                   = "none",
        dataloader_num_workers      = 2,
        ddp_find_unused_parameters  = False,
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer, model=trial_model,
        label_pad_token_id=-100,
        pad_to_multiple_of=8 if FP16 else None,
    )

    trainer = Seq2SeqTrainer(
        model            = trial_model,
        args             = trial_args,
        train_dataset    = train_dataset,
        eval_dataset     = val_dataset,
        processing_class = tokenizer,
        data_collator    = data_collator,
        callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)],
    )

    t0           = time.time()
    train_result = trainer.train()
    elapsed      = time.time() - t0

    best_eval_loss = trainer.state.best_metric
    if best_eval_loss is None:
        logs = [l for l in trainer.state.log_history if "eval_loss" in l]
        best_eval_loss = logs[-1]["eval_loss"] if logs else float("nan")

    print(f"  ✓ Done in {elapsed:.0f}s")
    print(f"  train_loss     : {train_result.training_loss:.4f}")
    print(f"  best_eval_loss : {best_eval_loss:.4f}")

    # ── Inference eval ────────────────────────────────────────────────────────
    print(f"  Running inference eval (beams={NUM_BEAMS}, length_penalty={LENGTH_PENALTY})...")
    metrics = _compute_metrics(model=trial_model)

    result = {
        "label"               : label,
        "description"         : cfg["description"],
        "use_dora"            : use_dora,
        "unfreeze_embeddings" : unfreeze_embeddings,
        "best_eval_loss"      : round(best_eval_loss, 4),
        "train_loss"          : round(train_result.training_loss, 4),
        "elapsed_s"           : round(elapsed, 1),
        **metrics,
    }
    DORA_RESULTS.append(result)

    print(f"  BLEU={metrics['sacrebleu']:.2f}  chrF={metrics['chrf']:.2f}  "
          f"METEOR={metrics['meteor']:.2f}  WER={metrics['wer']:.2f}%")

    del trial_model, trainer
    torch.cuda.empty_cache()
    return result


def _compute_metrics(model, num_beams=None, length_penalty=None, inference_batch=16):
    """Beam-search on val set, return dict of all metrics."""
    if num_beams is None:      num_beams      = NUM_BEAMS
    if length_penalty is None: length_penalty = LENGTH_PENALTY

    import sacrebleu as sb
    from rouge_score import rouge_scorer as rs_mod
    from nltk.translate.meteor_score import meteor_score as ms_fn
    from jiwer import wer as wer_fn, cer as cer_fn

    model.eval()
    eng_token_id       = tokenizer.convert_tokens_to_ids(TGT_LANG)
    val_sources        = df_val[DIALECT_COL_VAL].tolist()
    val_refs           = df_val[ENGLISH_COL_VAL].tolist()
    tokenizer.src_lang = SRC_LANG
    predictions        = []

    for i in range(0, len(val_sources), inference_batch):
        batch = val_sources[i : i + inference_batch]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_SOURCE_LEN,
        ).to(device)
        with torch.no_grad():
            generated = model.generate(
                **inputs,
                forced_bos_token_id=eng_token_id,
                max_length=MAX_TARGET_LEN,
                num_beams=num_beams,
                length_penalty=length_penalty,
                early_stopping=True,
            )
        predictions.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))

    refs_clean  = [r.strip() for r in val_refs]
    preds_clean = [p.strip() for p in predictions]

    bleu  = sb.corpus_bleu(preds_clean,  [refs_clean]).score
    chrf  = sb.corpus_chrf(preds_clean,  [refs_clean], word_order=2).score

    scorer_r = rs_mod.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=False)
    r1, r2, rL = [], [], []
    for p, r in zip(preds_clean, refs_clean):
        sc = scorer_r.score(r, p)
        r1.append(sc["rouge1"].fmeasure)
        r2.append(sc["rouge2"].fmeasure)
        rL.append(sc["rougeL"].fmeasure)

    met_scores = []
    for p, r in zip(preds_clean, refs_clean):
        pt, rt = p.split(), r.split()
        met_scores.append(ms_fn([rt], pt) if pt and rt else 0.0)

    wer_score = wer_fn(refs_clean, preds_clean) * 100
    cer_score = cer_fn(refs_clean, preds_clean) * 100

    return {
        "sacrebleu" : round(bleu,              4),
        "chrf"      : round(chrf,              4),
        "rouge1"    : round(np.mean(r1) * 100, 4),
        "rouge2"    : round(np.mean(r2) * 100, 4),
        "rougeL"    : round(np.mean(rL) * 100, 4),
        "meteor"    : round(np.mean(met_scores) * 100, 4),
        "wer"       : round(wer_score,         4),
        "cer"       : round(cer_score,         4),
    }

print("run_trial() and _compute_metrics() defined.")


run_trial() and _compute_metrics() defined.


## Trial A — Baseline (LoRA only, frozen embeddings)

`use_dora=False` — standard LoRA with the best Sylhet config.
This is your reference point.


In [25]:
# ════════════════════════════════════════════════════════════════════════════════
# TRIAL A — Baseline
# ════════════════════════════════════════════════════════════════════════════════
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

result_A = run_trial(GRID[0])
print(f"\nA done. BLEU={result_A['sacrebleu']:.2f}  chrF={result_A['chrf']:.2f}")



  TRIAL [A_baseline]  —  LoRA only — no DoRA, frozen embeddings
  use_dora=False   unfreeze_embeddings=False


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

  Trainable: 4,718,592  (0.761%)


Epoch,Training Loss,Validation Loss
1,8.429805,2.242188
2,7.028398,1.785156
3,6.440469,1.607422
4,6.181133,1.435547
5,5.930469,1.394531


  ✓ Done in 1076s
  train_loss     : 6.8972
  best_eval_loss : 1.3945
  Running inference eval (beams=5, length_penalty=0.8)...
  BLEU=49.51  chrF=65.91  METEOR=68.35  WER=38.55%

A done. BLEU=49.51  chrF=65.91


## Trial B — DoRA (frozen embeddings)

`use_dora=True` decomposes each LoRA weight update into a **magnitude** and a
**direction** component, updated separately. This better approximates full fine-tuning
and typically gives +1–4 BLEU on low-resource translation tasks.

The only code change vs Trial A is `use_dora=True` in `LoraConfig`.


In [26]:
# ════════════════════════════════════════════════════════════════════════════════
# TRIAL B — DoRA
# ════════════════════════════════════════════════════════════════════════════════
result_B = run_trial(GRID[1])
print(f"\nB done. BLEU={result_B['sacrebleu']:.2f}  "
      f"(vs A: {result_B['sacrebleu'] - result_A['sacrebleu']:+.2f})")



  TRIAL [B_dora]  —  DoRA — frozen embeddings
  use_dora=True   unfreeze_embeddings=False


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

  Trainable: 4,866,048  (0.785%)


Epoch,Training Loss,Validation Loss
1,8.426602,2.244141
2,7.019883,1.781250
3,6.425703,1.581055
4,6.161055,1.433594
5,5.917109,1.400391


  ✓ Done in 1499s
  train_loss     : 6.8836
  best_eval_loss : 1.4004
  Running inference eval (beams=5, length_penalty=0.8)...
  BLEU=51.72  chrF=67.21  METEOR=68.76  WER=36.87%

B done. BLEU=51.72  (vs A: +2.22)


## Results — A vs B

Run this cell after both trials complete.


In [27]:
# ════════════════════════════════════════════════════════════════════════════════
# DoRA A/B RESULTS — SYLHET
# ════════════════════════════════════════════════════════════════════════════════
import pandas as pd, os, json

df_res   = pd.DataFrame(DORA_RESULTS)
baseline = df_res[df_res["label"] == "A_baseline"].iloc[0]
dora_row = df_res[df_res["label"] == "B_dora"].iloc[0]

metric_cols = ["sacrebleu", "chrf", "rouge1", "rouge2", "rougeL", "meteor", "wer", "cer"]
cols_show   = ["label", "use_dora", "best_eval_loss", "train_loss"] + metric_cols + ["elapsed_s"]
cols_show   = [c for c in cols_show if c in df_res.columns]

print("\n" + "═"*65)
print(f"  DoRA A/B — SYLHET")
print("═"*65)
print(df_res[cols_show].to_string(index=False))

print("\n  DELTA  B − A  (+ve = DoRA better; for WER/CER −ve = better)")
for m in metric_cols:
    d   = dora_row[m] - baseline[m]
    tag = "✓" if (d > 0 and m not in ("wer","cer")) or (d < 0 and m in ("wer","cer")) else "✗"
    print(f"    {tag}  {m:<12}: {d:+.4f}")

bleu_delta  = dora_row["sacrebleu"] - baseline["sacrebleu"]
WINNER_DORA = bool(bleu_delta > 0)

print("\n" + "═"*65)
print("  VERDICT")
print("═"*65)
if WINNER_DORA:
    print(f"  ✅  DoRA HELPS  (+{bleu_delta:.2f} BLEU) → use use_dora=True in final run")
else:
    print(f"  ❌  DoRA does NOT help  ({bleu_delta:.2f} BLEU) → keep use_dora=False")

print(f"\n  Recommended final config for Sylhet 30-epoch run:")
print(f"    LEARNING_RATE        = {LEARNING_RATE}")
print(f"    lr_scheduler_type    = '{LR_SCHEDULER}'")
print(f"    LORA_TARGET_MODULES  = {LORA_TARGET_MODULES}")
print(f"    NUM_BEAMS            = {NUM_BEAMS}")
print(f"    length_penalty       = {LENGTH_PENALTY}")
print(f"    use_dora             = {WINNER_DORA}")

# Save results
os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = f"{OUTPUT_DIR}/dora_ab_results.json"
with open(out_path, "w") as f:
    json.dump({
        "dialect" : "Sylhet",
        "trials"  : DORA_RESULTS,
        "winner"  : {
            "use_dora"        : WINNER_DORA,
            "bleu_delta_vs_A" : round(bleu_delta, 4),
        },
    }, f, indent=2, default=str)
print(f"\n  Results saved → {out_path}")
print("═"*65)



═════════════════════════════════════════════════════════════════
  DoRA A/B — SYLHET
═════════════════════════════════════════════════════════════════
     label  use_dora  best_eval_loss  train_loss  sacrebleu    chrf  rouge1  rouge2  rougeL  meteor     wer     cer  elapsed_s
A_baseline     False          1.3945      6.8972    49.5066 65.9053 72.7459 57.4170 72.0595 68.3497 38.5542 29.3299     1076.2
    B_dora      True          1.4004      6.8836    51.7234 67.2084 73.4966 59.2637 72.7648 68.7555 36.8675 27.4227     1498.6

  DELTA  B − A  (+ve = DoRA better; for WER/CER −ve = better)
    ✓  sacrebleu   : +2.2168
    ✓  chrf        : +1.3031
    ✓  rouge1      : +0.7507
    ✓  rouge2      : +1.8467
    ✓  rougeL      : +0.7053
    ✓  meteor      : +0.4058
    ✓  wer         : -1.6867
    ✓  cer         : -1.9072

═════════════════════════════════════════════════════════════════
  VERDICT
═════════════════════════════════════════════════════════════════
  ✅  DoRA HELPS  (+2.22 BLEU